# Eye-state CNN — training in Google Colab (Stage 7)

This notebook is a thin driver around `training/train_eye_cnn.py`, so the training logic exists in exactly one place and runs the same way here (GPU) and locally (CPU smoke tests).

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4 is plenty).
2. Google Drive must contain `MyDrive/AI-Drowsiness-Detection/mrl_prepared/{train,val,test}.npz`, produced by `training/prepare_mrl.py` (Stage 6) from the MRL Eye Dataset. The split is subject-independent; the script re-checks that before training and refuses to run if any subject appears in two splits.
3. The repository must be reachable: either public, or add a GitHub token as a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar).

Outputs (curves, checkpoints, metrics, the final `eye_cnn.pt`) are written to Drive so nothing is lost when the runtime disconnects. Nothing in this notebook invents numbers: every metric comes from the test split of unseen subjects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/AI-Drowsiness-Detection'
DATA_DIR   = f'{DRIVE_ROOT}/mrl_prepared'        # train.npz / val.npz / test.npz from prepare_mrl.py
RUN_NAME   = 'run01'                              # change for each experiment (ablations get their own name)
RUN_DIR    = f'{DRIVE_ROOT}/runs/{RUN_NAME}'
EXPORT     = f'{DRIVE_ROOT}/models/eye_cnn.pt'    # final model; copy into the repo's models/ afterwards
EPOCHS     = 30

import os
for name in ('train', 'val', 'test'):
    path = f'{DATA_DIR}/{name}.npz'
    print(('OK      ' if os.path.exists(path) else 'MISSING ') + path)

In [ ]:
# Get the code. Public repo: plain clone. Private repo: token from Colab secrets (never paste a token into a cell).
import os, subprocess
REPO = 'ShivanshSatija/AI-Drowsiness-Detection'
if not os.path.exists('/content/AI-Drowsiness-Detection'):
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        url = f'https://{token}@github.com/{REPO}.git'
    except Exception:
        url = f'https://github.com/{REPO}.git'
    subprocess.run(['git', 'clone', '--depth', '1', url, '/content/AI-Drowsiness-Detection'], check=True)
%cd /content/AI-Drowsiness-Detection
!git log -1 --format='code at commit %h  %s'

# Training needs only numpy, OpenCV, matplotlib and torch - all preinstalled in Colab.
import torch, cv2, numpy
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to GPU')

In [ ]:
# Optional: build the splits here if they are not on Drive yet (requires the MRL zip on Drive).
# Skip this cell when mrl_prepared/ already exists.
MRL_ZIP = f'{DRIVE_ROOT}/mrl/mrlEyes_2018_01.zip'
if not os.path.exists(f'{DATA_DIR}/train.npz'):
    if os.path.exists('training/prepare_mrl.py'):
        !python training/prepare_mrl.py --source "$MRL_ZIP" --out "$DATA_DIR"
    else:
        print('training/prepare_mrl.py is not in this checkout yet - finish Stage 6 first.')

In [ ]:
# Train. Checkpoints (last.pt, best.pt), curves and metrics land in RUN_DIR on Drive after every epoch,
# so a disconnected runtime can resume with --resume "$RUN_DIR/last.pt".
!python training/train_eye_cnn.py --data "$DATA_DIR" --out "$RUN_DIR" --export "$EXPORT" --epochs $EPOCHS --workers 2

In [ ]:
from IPython.display import Image, Markdown, display
display(Image(f'{RUN_DIR}/curves.png'))
display(Image(f'{RUN_DIR}/confusion_matrix.png'))
display(Markdown(open(f'{RUN_DIR}/metrics.md').read()))

## Afterwards

1. Download `models/eye_cnn.pt` from Drive into the repository's `models/` folder and commit it together with `runs/<RUN_NAME>/metrics.json`, `metrics.md`, `curves.png` and `confusion_matrix.png` copied into `evaluation/results/`.
2. Report the numbers exactly as printed. The roadmap's > 95 % target is a target, not a requirement to be engineered; whatever the unseen-subject test set gives is the result.
3. Ablations: re-run with `--no-augment` or `--equalize` under a different `RUN_NAME` and compare `metrics.md` files.